RAG - Retrieval Augmented Generation is a technique used to make chatbots and LLMs answer question using external knowledge, instead of relying only on what was learnt during the training.

LLM + search engine + knowledge base = RAG

Why do we need RAG?

1. a normal LLM will only know what is it learnt during the training, also it may only have outdated information, and also it can hallucinate while giving answers(make up facts).

2. RAG chatbot - search the relevant documents first, use those documents to generate answers and provide more accurate and up to date response.

How RAG works?

step 1: Example: What is the companies leave policy.
Here the LLM doesn't know your companies internal policy.

step 2: Retrieve relevant documents. The system searches a knowledge base containing:

1. PDFs

2. Documents

3. Databases

4. Website content

5. FAQs

Step 3: Augment the prompt

The chatbot will now create a new prompt.
- Employees are entitled to 26 weeks of maternity leave

Step 4: Generate the answer

An LLM uses the retrieved context to answer:
According to company policy, employees are entitled to 26 weeks of paid maternity leave.



Understanding the workflow of RAG systems:

1. User question
2. embedding model
3. vector database (Pinecone, Chroma, FAISS, Weaviate)
4. retrieve top k chunks
5. prompt construction
6. LLM (GPT, Claude etc)
7. Answer

RAG vs Fine tuning
- feature -
1. Uses latest data - RAG is better
2. easy to update - RAG is better
3. Cost - RAG will have LOW cost and fine tuning will have high cost.
4. company documents - RAG is better
5. changes model behaviour -  fine tuning is better.



In [1]:
# installing dependencies
!pip install sentence-transformers
!pip install faiss-cpu
!pip install langchain
!pip install langchain-community
!pip install langchain-huggingface
!pip install pypdf2

In [2]:
import os
import warnings
warnings.filterwarnings('ignore')

#RAG component 1 - text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

#RAG component 2 - Embedding model
from langchain_huggingface import HuggingFaceEmbeddings

#RAG component 3 - Vector store
from langchain_community.vectorstores import FAISS

#RAG component 4 - Document Schema
from langchain_core.documents import Document

#RAG component 5 - Prompt template
from langchain_core.prompts import PromptTemplate

In [3]:
# step 3 - knowledge base
# knowledge base is the collection of documents the chatbot will search through

DATA_OPTION = 'A' # Change this accordingly A - built in text, B is for Pasting your Text and C is for Documents

SAMPLE_TEXT = """
Artificial Intelligence (AI) refers to the simulation of human intelligence in machines.
AI systems can learn, reason, plan, perceive, and understand language.

Machine Learning (ML) is a subset of AI that enables systems to learn from data without
being explicitly programmed. ML algorithms build models from training examples to make
predictions or decisions on new data.

Deep Learning is a further subset of ML that uses artificial neural networks with many
layers. It has achieved breakthroughs in image recognition, speech recognition, and NLP.

Natural Language Processing (NLP) allows computers to understand, interpret, and generate
human language. Applications include chatbots, translation, and sentiment analysis.

Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model.
Instead of relying solely on what the model memorised during training, RAG fetches
relevant passages from a knowledge base at query time and uses them as context.
This dramatically reduces hallucinations and keeps answers grounded in real sources.

FAISS (Facebook AI Similarity Search) is a library for efficient similarity search over
dense vectors. It can search millions of vectors in milliseconds on a CPU.

Embeddings are numerical representations of text (vectors of floating-point numbers).
Similar sentences have similar vectors — this is how a vector store finds relevant chunks.

A Large Language Model (LLM) is a neural network trained on massive text datasets.
LLMs can generate fluent, coherent text and follow complex instructions.
Examples include GPT-4 (OpenAI), Gemini (Google), and Claude (Anthropic).

Hallucination in AI refers to when a language model generates information that sounds
plausible but is factually incorrect or invented. RAG helps reduce this by grounding
the model's response in retrieved, verifiable source text.

The NSE (National Stock Exchange of India) is one of the largest stock exchanges in the
world. It provides a platform for trading equities, derivatives, and debt instruments.
Market microstructure refers to the study of how trading mechanisms, order types, and
participant behaviour affect price formation and liquidity.

Prompt engineering is the practice of carefully designing inputs to language models to
obtain desired outputs. In RAG, the prompt includes both retrieved context and the user
question, guiding the LLM to answer only from what was retrieved.
"""

MY_CUSTOM_TEXT = """
Replace this with custome TEXT """

UPLOADED_FILE_PATH = ''

raw_documents = []

if DATA_OPTION == 'A':
    raw_documents = [Document(page_content = SAMPLE_TEXT, metadata = {'source': 'Built in AI SAMPLE'})]

elif DATA_OPTION == 'B':
    raw_documents = [Document(page_content = MY_CUSTOM_TEXT, metadata = {'source': 'custom_text'})]

else:
    from langchain_community.document_loaders import PyPDF2Loader
    loader = PyPDF2Loader(UPLOADED_FILE_PATH)


# STEP 4 text Splitter

Why do we need to split the text

 LLM have limited text window - the amount of text they can read at once, we can't feed the entire document into every query

 instead we,
 1. split the document into small chunks
 2. at query time, fetch only the most relevant chunks

 the overlap ensures sentence split across 2 chunk boundaries doesn't lose meaning



In [4]:
CHUNK_SIZE = 400
CHUNK_OVERLAP = 50

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = CHUNK_SIZE,
    chunk_overlap = CHUNK_OVERLAP,
    separators = ['\n\n','\n', '. ',' ',''],
    length_function = len
)

chunks = text_splitter.split_documents(raw_documents)

for i, chunk in enumerate(chunks[:3]):
    print(chunk.page_content.strip())


Artificial Intelligence (AI) refers to the simulation of human intelligence in machines.
AI systems can learn, reason, plan, perceive, and understand language.

Machine Learning (ML) is a subset of AI that enables systems to learn from data without
being explicitly programmed. ML algorithms build models from training examples to make
predictions or decisions on new data.
Deep Learning is a further subset of ML that uses artificial neural networks with many
layers. It has achieved breakthroughs in image recognition, speech recognition, and NLP.

Natural Language Processing (NLP) allows computers to understand, interpret, and generate
human language. Applications include chatbots, translation, and sentiment analysis.
Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model.
Instead of relying solely on what the model memorised during training, RAG fetches
relevant passages from a knowledge base at query time and uses them as context.
This dramatically reduce

#embedding model
This converts text into a list of numbers (a vector) that captures the meaning of the text.

When you ask a question, question is turned into a embedded vector, FAISS then finds the chunk vectors closest to your question vector those are the relevant chunks.

In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs = {'device': 'cpu'},
    #encode_kwargs = {'normalize_embedding': True}
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
# step 6 - creating vector store (FAISS)

# what is vector store -
# it is a special database that stores all chunk embeddings (vectors)
# searches them in milliseconds using nearest neighbour algorithm

VECTOR_STORE_PATH = './rag_index' #saved to disk

index_file = os.path.join(VECTOR_STORE_PATH, 'index.faiss')
if os.path.exists(index_file):
    vectorstore = FAISS.load_local(
    VECTOR_STORE_PATH,
    embedding_model,
    allow_dangerous_deserialization = True
)
else:
    vectorstore = FAISS.from_documents(chunks, embedding_model)
    os.makedirs(VECTOR_STORE_PATH, exist_ok = True)
    vectorstore.save_local(VECTOR_STORE_PATH)




In [7]:
# Retriever

TOP_K = 3

retriever = vectorstore.as_retriever(search_type = 'similarity', search_kwargs={'k': TOP_K})

test_question = ["What is hallucination"]

retrieved = retriever.invoke(test_question[0])

retrieved[0]


Document(id='8b478473-09fc-49c8-b789-014873b3d401', metadata={'source': 'Built in AI SAMPLE'}, page_content="Hallucination in AI refers to when a language model generates information that sounds\nplausible but is factually incorrect or invented. RAG helps reduce this by grounding\nthe model's response in retrieved, verifiable source text.")

In [8]:
# prompt template
# it is an instruction card you hand to the LLM, it tells the model what role does it play, what context to use and how to behave

RAG_TEMPLATE = """
You are a helpful and accurate teaching assistant,
USE only the context provided below to answer the questions
If the answer is not in the context say:
"i dont have enough information in the knowledge base to answer that"
Do not make up facts.

Context:
{context}

Question: {question}
Answer:
"""

#rag_prompt = PromptTemplate(input_variable = ['context', 'question'], template = RAG_TEMPLATE)
rag_prompt = PromptTemplate(input_variables = ['context', 'question'], template = RAG_TEMPLATE)

In [9]:
!pip install -U transformers langchain-huggingface langchain-community

In [12]:
# LLM
#LLM_CHOICE = 'local'

#if LLM_CHOICE == 'local':
    #from langchain_community.llms import HuggingFacePipeline
from langchain_huggingface import HuggingFacePipeline
    #from transformers import pipeline as hf_pipeline
from transformers import pipeline
    #pipe = hf_pipeline('text-generation', model = 'google/flan-t5-base', max_new_tokens = 256)
pipe = pipeline("text-generation", model = "Qwen/Qwen2.5-0.5B-Instruct", max_new_tokens = 256, do_sample=False,
return_full_text=False)
llm = HuggingFacePipeline(pipeline = pipe)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [13]:
#RAG pipeline
from langchain_classic.chains import RetrievalQA

rag_pipeline = RetrievalQA.from_chain_type(
    llm = llm,
    chain_type = 'stuff',
    retriever = retriever,
    return_source_documents = True,
    chain_type_kwargs = {'prompt': rag_prompt}
)

In [14]:
def ask(question, show_sources = True, show_prompt = True):
    result = rag_pipeline.invoke({'query': question})
    answer = result.get('result','').strip()
    source_documents = result.get('source_documents', [])

    return answer

In [15]:
print(ask("What is Retrieval-augmented Generation?"))

Retrieval-Augmented Generation (RAG) combines a retrieval system with a language model.
Instead of relying solely on what the model memorised during training, RAG fetches
relevant passages from a knowledge base at query time and uses them as context.
This dramatically reduces hallucinations and keeps answers grounded in real sources.
The prompt engineering is the practice of carefully designing inputs to language models to
obtain desired outputs. In RAG, the prompt includes both retrieved context and the user
question, guiding the LLM to answer only from what was retrieved.
FAISS (Facebook AI Similarity Search) is a library for efficient similarity search over
dense vectors. It can search millions of vectors in milliseconds on a CPU.
What is the answer? The answer is:
Retrieval-Augmented Generation (RAG). This technology combines a retrieval system with a language model to reduce hallucinations and keep answers grounded in real sources. It achieves this by fetching relevant passages fr